# 1차 전처리 — 영문 텍스트 (냄새 관련 데이터)

**목적**: 원본 CSV(본문 + 댓글 컬럼 분리)를 받아 단계별로 정제한 뒤 `ENG_1st_contents.csv`로 저장합니다.

**파이프라인**
1. 원본 CSV 로드
2. 본문 + 댓글 컬럼을 단일 `contents` 컬럼으로 결합
3. URL 제거
4. 30자 이하 텍스트 제거
5. 중복 제거
6. 스팸/광고 제거 (`blockenters/sms-spam-classifier`)
7. AI 생성 텍스트 제거 (`fakespot-ai/roberta-base-ai-text-detection-v1`)
8. 최종 저장 → `ENG_1st_contents.csv`

> 상단 설정 셀의 `RAW_CSV_PATH`, `POST_COL`, `COMMENT_COL` 만 변경하면 다른 데이터셋에도 동일하게 적용할 수 있습니다.

## 0. 의존성 설치 (필요 시 주석 해제)

In [1]:
!pip install -q pandas transformers torch tqdm

## 1. Imports

In [2]:
import re
import os
import gc
from pathlib import Path

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import pipeline

tqdm.pandas()

# DCX project root (parent of text_preprocessing/)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "text_preprocessing":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT / "text_preprocessing").exists():
    PROJECT_ROOT = PROJECT_ROOT

DATA_DIR = PROJECT_ROOT / "data" / "text_preprocessing"
OUT_DIR = PROJECT_ROOT / "out" / "text_preprocessing"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. 설정

- `RAW_CSV_PATH`: 원본 CSV 경로 (본문 + 댓글 컬럼이 분리된 파일)
- `POST_COL`, `COMMENT_COL`: 원본의 본문 컬럼명, 댓글 컬럼명
- `OUTPUT_CSV`: 결과 파일 경로
- 장비 자동 감지 (CUDA → MPS → CPU)

In [ ]:
RAW_CSV_PATH = DATA_DIR / "raw_data.csv"  # TODO: 실제 원본 경로로 교체
TITLE_COL = "title"                       # TODO: 원본의 제목 컬럼명
CONTENT_COL = "content"                   # TODO: 원본의 본문 컬럼명
COMMENT_COL = "comment"                   # TODO: 원본의 댓글 컬럼명

OUTPUT_CSV = OUT_DIR / "ENG_1st_contents.csv"

MIN_CHAR_LEN = 30
BATCH_SIZE = 64

if torch.cuda.is_available():
    device = 0
    device_str = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = -1
    device_str = "mps"
else:
    device = -1
    device_str = "cpu"

print(f"project root: {PROJECT_ROOT}")
print(f"device: {device_str}")

device: cpu


## Step 1 — 원본 CSV 로드

In [ ]:
raw_df = pd.read_csv(RAW_CSV_PATH)
print("shape:", raw_df.shape)
print("columns:", list(raw_df.columns))
raw_df.head()

## Step 2 — 본문 + 댓글을 단일 `contents` 컬럼으로 결합

본문 컬럼과 댓글 컬럼의 값을 각각 `contents` 이라는 단일 컬럼으로 통일합니다. NaN/빈 문자열은 제거.

In [ ]:
titles = raw_df[TITLE_COL].dropna().astype(str)
contents = raw_df[CONTENT_COL].dropna().astype(str)
comments = raw_df[COMMENT_COL].dropna().astype(str)

df = pd.concat(
    [
        pd.DataFrame({"contents": titles.values, "source": "title"}),
        pd.DataFrame({"contents": contents.values, "source": "content"}),
        pd.DataFrame({"contents": comments.values, "source": "comment"}),
    ],
    ignore_index=True,
)

df["contents"] = df["contents"].astype(str).str.strip()
df = df[df["contents"].str.len() > 0].reset_index(drop=True)

print("after merge:", df.shape)
df.head()

## Step 3 — URL 제거

`http(s)://...`, `www....` 패턴과 연속 공백을 정리합니다.

In [ ]:
URL_RE = re.compile(r"http\S+|www\.\S+", flags=re.IGNORECASE)
WS_RE = re.compile(r"\s+")

def remove_url(text: str) -> str:
    text = URL_RE.sub(" ", text)
    text = WS_RE.sub(" ", text).strip()
    return text

df["contents"] = df["contents"].progress_apply(remove_url)
print("after URL removal:", df.shape)
df.head()

## Step 4 — 30자 이하 제거

In [ ]:
before = len(df)
df = df[df["contents"].str.len() >= MIN_CHAR_LEN].reset_index(drop=True)
print(f"length filter (>= {MIN_CHAR_LEN} chars): {before} → {len(df)}")

## Step 5 — 중복 제거

In [ ]:
before = len(df)
df = df.drop_duplicates(subset="contents").reset_index(drop=True)
print(f"deduplicate: {before} → {len(df)}")

## Step 5-1 — ㅋㅋ, ㅎㅎ 제거 (한국어 전용)

In [ ]:
# ㅋ, ㅎ가 2번 이상 반복되는 표현 제거
df["contents"] = (
    df["contents"]
    .astype(str)
    .str.replace(r"[ㅋㅎ]{2,}", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

## Step 6 — 스팸/광고 분류기 로드

`blockenters/sms-spam-classifier` 사용. 라벨 매핑을 먼저 확인.

In [ ]:
spam_clf = pipeline(
    "text-classification",
    model="blockenters/sms-spam-classifier",
    truncation=True,
    device=device,
)

print("id2label:", spam_clf.model.config.id2label)

## Step 7 — 스팸 분류 실행 (배치)

`tqdm`으로 진행률을 표시하면서 배치 단위로 추론. 이 모델은 `LABEL_0 = ham`, `LABEL_1 = spam` 입니다.

In [ ]:
texts = df["contents"].tolist()
spam_labels, spam_scores = [], []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="spam classify"):
    batch = texts[i : i + BATCH_SIZE]
    preds = spam_clf(batch, batch_size=BATCH_SIZE, truncation=True)
    spam_labels.extend([p["label"] for p in preds])
    spam_scores.extend([p["score"] for p in preds])

df["spam_label"] = spam_labels
df["spam_score"] = spam_scores
df["spam_label"].value_counts()

## Step 8 — 스팸 필터링

`LABEL_0`(ham) 인 행만 유지.

In [ ]:
before = len(df)
df = df[df["spam_label"] == "LABEL_0"].reset_index(drop=True)
print(f"spam filter: {before} → {len(df)}")

del spam_clf
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Step 9 — AI 생성 텍스트 감지기 로드 (외국어 전용)

`fakespot-ai/roberta-base-ai-text-detection-v1` 사용. 라벨 매핑 확인.
어마 무시하게 양이 줄 수 있으니 주의의

In [ ]:
ai_clf = pipeline(
    "text-classification",
    model="fakespot-ai/roberta-base-ai-text-detection-v1",
    truncation=True,
    device=device,
)

print("id2label:", ai_clf.model.config.id2label)

## Step 10 — AI 감지 실행 (배치)

In [ ]:
texts = df["contents"].tolist()
ai_labels, ai_scores = [], []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="AI detect"):
    batch = texts[i : i + BATCH_SIZE]
    preds = ai_clf(batch, batch_size=BATCH_SIZE, truncation=True)
    ai_labels.extend([p["label"] for p in preds])
    ai_scores.extend([p["score"] for p in preds])

df["ai_label"] = ai_labels
df["ai_score"] = ai_scores
df["ai_label"].value_counts()

## Step 11 — AI 텍스트 필터링

`Human` 라벨을 가진 행만 유지. 라벨명이 다르면(예: `LABEL_0`/`LABEL_1`) `id2label` 출력을 보고 `HUMAN_LABEL` 값을 조정하세요.

In [ ]:
HUMAN_LABEL = "Human"  # 필요 시 모델 id2label에 맞게 수정

before = len(df)
df = df[df["ai_label"] == HUMAN_LABEL].reset_index(drop=True)
print(f"AI text filter (keep {HUMAN_LABEL}): {before} → {len(df)}")

del ai_clf
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Step 12 — 최종 저장

`contents` 컬럼만 남겨 `ENG_1st_contents.csv`로 저장합니다. 분류 메타(라벨/스코어)도 함께 저장하고 싶으면 아래 주석 해제.

In [ ]:
out_df = df[["contents"]].copy()
# out_df = df[["contents", "source", "spam_label", "spam_score", "ai_label", "ai_score"]].copy()

out_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"saved → {OUTPUT_CSV} ({len(out_df)} rows)")

## 요약

- 본문 + 댓글을 단일 `contents` 컬럼으로 통합
- URL / 30자 미만 / 중복 제거
- 스팬 분류기(`blockenters/sms-spam-classifier`)로 광고·스팸 제거
- AI 감지기(`fakespot-ai/roberta-base-ai-text-detection-v1`)로 AI 생성 내용 제거
- 결과: `ENG_1st_contents.csv` → 2차 전처리 노트북의 입력으로 사용